In [12]:
!pip install -U langchain langchain-huggingface langchain-community langchain-classic faiss-cpu tiktoken docx2txt pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 16.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 30.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 27.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 31.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 MB 31.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 32.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 37.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 21.2 MB/s  0:00:00
  Attempting uninstall: tokenizers━━━━╸━━━━━━━━━ 17/22 [rich]]]learn]
    Found existing installation: tokenizers 0.23.190m━━━━━━━━━ 17/22 [rich]
    Uninstalling tokenizers-0.23.1:╸━━━━━━━━━ 17/

In [7]:
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

In [38]:
from pathlib import Path

pdf_path = Path("Employee_Rights_Charter.pdf")
docx_path = Path("Company_Policy_Manual.docx")

if not pdf_path.exists() or not docx_path.exists():
    raise FileNotFoundError("Make sure both files are in your RAG folder.")

pdf_docs = PyPDFLoader(str(pdf_path)).load()
docx_docs = Docx2txtLoader(str(docx_path)).load()

all_docs = pdf_docs + docx_docs
print(f"Loaded pages/docs: PDF={len(pdf_docs)}, DOCX={len(docx_docs)}, TOTAL={len(all_docs)}")

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False,
 )

chunks = text_splitter.split_documents(all_docs)
print(f"Created chunks: {len(chunks)}")
print("Sample chunk metadata:", chunks[0].metadata if chunks else "No chunks")

Loaded pages/docs: PDF=2, DOCX=1, TOTAL=3
Created chunks: 10
Sample chunk metadata: {'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'source': 'Employee_Rights_Charter.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [39]:
# Build embeddings and store vectors in a FAISS index
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

# Quick sanity check
test_results = vectorstore.similarity_search("employee rights and workplace policy", k=2)
print(f"FAISS index saved to ./faiss_index")
print(f"Stored chunks: {vectorstore.index.ntotal}")
print("Top match source:", test_results[0].metadata.get("source") if test_results else "No results")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4825.72it/s]


FAISS index saved to ./faiss_index
Stored chunks: 10
Top match source: Employee_Rights_Charter.pdf


In [43]:
import os
from langchain_core.prompts import PromptTemplate

# Build retriever from the FAISS vector store created above
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

qa_prompt = PromptTemplate.from_template(
    """You are a precise HR policy assistant. Use only the provided context.
If the answer is not in context, say: I don't know based on the provided documents.
Return one short, direct answer sentence only.

Context:
{context}

Question: {question}
Answer:"""
)

# Try hosted HF inference first; if unavailable, fall back to a local instruction model
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
llm = None

if hf_token:
    try:
        llm = HuggingFaceEndpoint(
            repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            provider="hf-inference",
            task="text-generation",
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            return_full_text=False,
            huggingfacehub_api_token=hf_token,
        )
        _ = llm.invoke("Hello")
        print("Using hosted Hugging Face inference.")
    except Exception as e:
        llm = None
        print("Hosted inference unavailable, switching to local model.")
        print("Reason:", str(e)[:200])

if llm is None:
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline

    local_pipeline = pipeline(
        task="text-generation",
        model="Qwen/Qwen2.5-0.5B-Instruct",
        do_sample=False,
        max_new_tokens=128,
        return_full_text=False,
    )
    llm = HuggingFacePipeline(pipeline=local_pipeline)
    print("Using local model: Qwen/Qwen2.5-0.5B-Instruct")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": qa_prompt},
)

print("RAG chain is ready. Use qa_chain.invoke({'query': 'your question'})")

Hosted inference unavailable, switching to local model.
Reason: (Request ID: Root=1-6a6ac3b4-174eff50032c488a4d8e8157;e9f2b967-d008-4e27-ace7-8f5e6f29b38c)

403 Forbidden: This authentication method does not have sufficient permissions to call Inference Providers 


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7180.20it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Using local model: Qwen/Qwen2.5-0.5B-Instruct
RAG chain is ready. Use qa_chain.invoke({'query': 'your question'})


In [41]:
import os

# Optional: set token for this notebook session only (uncomment and replace).
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_your_token_here"

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if token:
    masked = f"{token[:6]}...{token[-4:]}" if len(token) > 12 else "[set]"
    print("Token is set in this notebook session.")
    print("Masked token:", masked)
else:
    print("Token is not set.")
    print("Set it in terminal before running RAG setup:")
    print("export HUGGINGFACEHUB_API_TOKEN='your_token_here'")

Token is set in this notebook session.
Masked token: hf_MyE...HVuP


In [46]:
import re

query = "give me 3 bullet points of the company policies that which is unfair to employees and should be changed"
result = qa_chain.invoke({"query": query})

raw_answer = result["result"].strip()
clean_answer = re.split(r"(?<=[.!?])\s+", raw_answer)[0].strip()
print("Answer:", clean_answer)

top_source = result.get("source_documents", [])
if top_source:
    src = top_source[0].metadata.get("source", "unknown")
    page = top_source[0].metadata.get("page", "n/a")
    print(f"Top source: {src}, page={page}")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: The Employee Bill of Rights does not mention any specific policies regarding unfair treatment of employees.
Top source: Employee_Rights_Charter.pdf, page=0
